[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C23_Frontier_Alignment_Course/04_weak_to_strong/04_weak_to_strong.ipynb)

# 04 · Weak-to-Strong 泛化（用 numpy 模拟）

目标：从零复现 **weak-to-strong** 现象——用弱老师的**带噪标签**训强学生, 强学生**超过老师**；算 **PGR**, 加 **辅助置信损失**, 做 **天花板分析**。

路线：真概念沙盒 → 弱老师带噪标签 → 强学生 w2s → PGR → 置信损失 → 天花板分析 → ✏️ 练习 → 📖 答案 → 🧪 Burns 胶囊。

> 心智模型：**弱老师标签 = 真信号(多数) + 零散错误(难样本); 强学生有强先验, 拟合信号而非噪声 → 超过老师。** 用合成真概念, 天然规避「测试泄漏」(强学生预训练时绝不可能见过我们临时造的概念)。

## 1 · 真概念沙盒（ground truth）

高维空间里设一个我们知道的**线性真概念**（一个超平面）。样本到边界的「间隔」(margin)决定难易：离边界远 = 容易、清晰；离边界近 = 困难、模糊。这后面决定了弱老师在哪里会标错。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

D = 20
w_true = rng.standard_normal(D); w_true /= np.linalg.norm(w_true)   # 真概念方向
def gen(n, rng):
    X = rng.standard_normal((n, D))
    margin = X @ w_true                 # 到真边界的间隔(带符号)
    y = (margin > 0).astype(int)        # 真标签
    return X, y, margin

Xtr, ytr, mtr = gen(4000, rng)
Xte, yte, mte = gen(4000, rng)
print('训练集', Xtr.shape, '测试集', Xte.shape)
print('正类比例(应≈0.5):', round(ytr.mean(), 3))
assert Xtr.shape == (4000, D) and set(np.unique(ytr)) == {0, 1}
print('✅ 真概念沙盒就绪：我们知道每个样本的真标签与难易(margin)')

## 2 · 弱老师：在困难样本上标错

弱老师不是随机乱标 —— 它在**清晰样本**(margin 大)上标对, 在**边界附近**(margin 小)上容易标错。
这模拟真实的弱监督者：简单的会、难的不会。它的标签是真标签的**带噪**版本。

In [ ]:
def weak_teacher_labels(margin, y, rng, noise=1.2):
    '''翻转概率随 |margin| 减小而增大：边界附近(难)更易标错。'''
    flip_p = 1.0 / (1.0 + np.exp(np.abs(margin) / noise)) * 0.9   # 难样本翻转概率高达~0.45
    flips = rng.random(len(y)) < flip_p
    return np.where(flips, 1 - y, y)

weak_tr = weak_teacher_labels(mtr, ytr, rng)
weak_te = weak_teacher_labels(mte, yte, np.random.default_rng(99))
weak_acc = (weak_te == yte).mean()
# 验证: 老师在清晰样本上更准, 在边界附近更差
easy = np.abs(mte) > 1.5; hard = np.abs(mte) < 0.3
acc_easy = (weak_te[easy] == yte[easy]).mean()
acc_hard = (weak_te[hard] == yte[hard]).mean()
print(f'弱老师整体准确率 = {weak_acc:.3f}')
print(f'  清晰样本上 = {acc_easy:.3f}  |  边界附近 = {acc_hard:.3f}')
assert 0.6 < weak_acc < 0.85, '弱老师应明显优于随机但远非完美'
assert acc_easy > acc_hard, '老师在清晰样本上更准(错误集中在难样本)'
print('✅ 弱老师带噪标签就绪：错误集中在边界附近的困难样本')

## 3 · weak-to-strong：强学生超过弱老师

**核心实验**：用弱老师的带噪标签训一个**强学生**（全特征 + 适当正则的 logistic 模型）, 在**真标签**上评估它。看它是否**超过弱老师**。

In [ ]:
def sigmoid(z): return 1.0/(1.0+np.exp(-np.clip(z, -30, 30)))
def predict(w, b, X): return (sigmoid(X @ w + b) > 0.5).astype(int)
def accuracy(w, b, X, y): return (predict(w, b, X) == y).mean()

def train_logreg(X, y, epochs=400, lr=0.5, l2=1e-2):
    n, d = X.shape; w = np.zeros(d); b = 0.0
    y = y.astype(float)
    for _ in range(epochs):
        p = sigmoid(X @ w + b); g = p - y
        w -= lr * (X.T @ g / n + l2 * w); b -= lr * g.mean()
    return w, b

# 强学生: 用弱标签训
ws, bs = train_logreg(Xtr, weak_tr)
student_acc = accuracy(ws, bs, Xte, yte)        # 在真标签上评估
print(f'弱老师准确率   = {weak_acc:.3f}')
print(f'强学生准确率   = {student_acc:.3f}  (用弱老师的带噪标签训出来的!)')
assert student_acc > weak_acc + 0.05, 'w2s: 强学生应显著超过弱老师'
print(f'\n✅ weak-to-strong 发生！强学生比教它的弱老师高 {student_acc-weak_acc:.3f}')
print('   机制: 强先验让学生拟合「简洁真规律」而非老师的「零散错误」')

## 4 · PGR：恢复了多少差距

$$\mathrm{PGR}=\frac{\text{强学生(弱标签)} - \text{弱老师}}{\text{强天花板(真标签)} - \text{弱老师}}$$

需要第三个角色：**强天花板** = 同样的强模型, 但用**真标签**训。PGR=0 只学到老师水平; PGR=1 完全恢复到天花板。

In [ ]:
# 强天花板: 用真标签训同样的模型
wc, bc = train_logreg(Xtr, ytr)
ceiling_acc = accuracy(wc, bc, Xte, yte)

def pgr(student, weak, ceiling):
    return (student - weak) / (ceiling - weak)

PGR = pgr(student_acc, weak_acc, ceiling_acc)
print(f'弱老师   = {weak_acc:.3f}  (PGR 的 0 分线)')
print(f'强学生   = {student_acc:.3f}')
print(f'强天花板 = {ceiling_acc:.3f}  (PGR 的满分线)')
print(f'PGR      = {PGR:.3f}  (恢复了弱老师到天花板差距的 {PGR:.0%})')
assert 0.0 < PGR <= 1.05, 'PGR 应落在 (0,1]'
assert ceiling_acc > student_acc > weak_acc, '天花板 > 学生 > 老师'
print('\n✅ PGR 量化了 w2s：弱监督引出了强模型的大部分(非全部)能力')

## 5 · 辅助置信损失：让学生敢偏离弱老师

$$\mathcal{L}=(1-\alpha)\,\mathrm{CE}(\hat y, \tilde y_{weak}) + \alpha\,\mathrm{CE}(\hat y, \hat y_{hardened})$$

第二项鼓励学生相信**自己的硬化预测**, 在自信处敢不同意弱老师。
**诚实提示**：logistic 回归本就抗噪, 玩具里置信损失对*准确率*提升有限; 我们验证它的**定义性机制**——让学生**更自信、更少盲从弱标签**。

In [ ]:
def train_with_confidence(X, y_weak, epochs=400, lr=0.5, l2=1e-2, alpha=0.8):
    n, d = X.shape; w = np.zeros(d); b = 0.0
    y_weak = y_weak.astype(float)
    for ep in range(epochs):
        p = sigmoid(X @ w + b)
        hardened = (p > 0.5).astype(float)              # 学生自己的硬化预测
        a = alpha * min(1.0, ep / (epochs * 0.5))       # α 逐渐增大: 先学老师再敢质疑
        target = (1 - a) * y_weak + a * hardened        # 混合: 弱标签 + 自信自我
        g = p - target
        w -= lr * (X.T @ g / n + l2 * w); b -= lr * g.mean()
    return w, b

wconf, bconf = train_with_confidence(Xtr, weak_tr)
# 定义性机制: 学生变得更自信(平均 |logit| 显著更大)
conf_plain = np.abs(Xte @ ws + bs).mean()
conf_aux   = np.abs(Xte @ wconf + bconf).mean()
# 学生与弱标签的总体一致率(置信损失让它更敢偏离弱标签 -> 一致率下降)
agree_plain = (predict(ws, bs, Xtr) == weak_tr).mean()
agree_aux   = (predict(wconf, bconf, Xtr) == weak_tr).mean()
print(f'平均置信度 |logit|:    普通 {conf_plain:.2f}  ->  置信损失 {conf_aux:.2f}')
print(f'与弱标签的一致率:      普通 {agree_plain:.3f}  ->  置信损失 {agree_aux:.3f}')
assert conf_aux > conf_plain * 1.5, '置信损失应让学生显著更自信'
assert agree_aux < agree_plain, '置信损失应让学生更敢偏离弱标签(一致率下降)'
# 学生准确率仍不低于弱老师(没把自己带偏)
assert accuracy(wconf, bconf, Xte, yte) > weak_acc, '加置信损失后仍保持 w2s'
print('\n✅ 置信损失的定义性机制成立：学生更自信、更敢偏离弱标签, 且仍超过弱老师')
print('   (真实大模型上, 这一机制转化为 Burns et al. 报告的 PGR 提升)')

## 6 · 天花板分析：老师越弱, PGR 越低

w2s 不是「成功/失败」的二元, 而是一条随师生能力差变化的曲线。
让弱老师**越来越弱**(增大噪声), 看 PGR 怎么变 —— 监督差到一定程度, 连强先验也救不回来。

In [ ]:
print(f"{'老师噪声':>8}{'弱老师acc':>10}{'强学生acc':>10}{'PGR':>8}")
pgrs = []
for noise in [0.6, 1.2, 2.5, 5.0]:
    wl_tr = weak_teacher_labels(mtr, ytr, np.random.default_rng(10), noise=noise)
    wl_te = weak_teacher_labels(mte, yte, np.random.default_rng(11), noise=noise)
    wacc = (wl_te == yte).mean()
    sw, sb = train_logreg(Xtr, wl_tr)
    sacc = accuracy(sw, sb, Xte, yte)
    pg = pgr(sacc, wacc, ceiling_acc)
    pgrs.append(pg)
    print(f'{noise:>8.1f}{wacc:>10.3f}{sacc:>10.3f}{pg:>8.3f}')
# 噪声越大(老师越弱), PGR 越低
assert pgrs[0] > pgrs[-1], '老师越弱, 能恢复的差距比例越低'
print('\n✅ 天花板分析：监督越弱, w2s 恢复得越少 —— 弱监督不能差到没有真信号')

---
## ✏️ 练习 1：实现 PGR `compute_pgr`

实现 `compute_pgr(student_acc, weak_acc, ceiling_acc)`：返回 PGR。
处理边界：当 `ceiling_acc == weak_acc`（无可恢复空间）时返回 0.0。

In [ ]:
def compute_pgr(student_acc, weak_acc, ceiling_acc):
    # TODO: (student-weak)/(ceiling-weak); 分母为0时返回 0.0
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert abs(compute_pgr(0.9, 0.7, 0.95) - 0.8) < 1e-9, '(0.9-0.7)/(0.95-0.7)=0.8'
assert compute_pgr(0.7, 0.7, 0.95) == 0.0, '学生=老师 -> PGR=0'
assert compute_pgr(0.95, 0.7, 0.95) == 1.0, '学生=天花板 -> PGR=1'
assert compute_pgr(0.8, 0.8, 0.8) == 0.0, '无空间 -> 0'
print('PGR(学生0.9,老师0.7,天花板0.95) =', compute_pgr(0.9, 0.7, 0.95))
print('✅ 练习 1 通过：PGR 计算正确(含边界)')

## ✏️ 练习 2：带噪弱标签生成 `make_weak_labels`

实现一个**更简单**的弱老师：以**固定**翻转概率 `flip_rate` 随机翻转真标签（均匀噪声）。
`make_weak_labels(y, flip_rate, rng)` 返回带噪标签。验证带噪标签准确率 ≈ 1 - flip_rate。

In [ ]:
def make_weak_labels(y, flip_rate, rng):
    # TODO: 以概率 flip_rate 翻转每个标签(1-y), 否则保留
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
y = (np.random.default_rng(5).random(5000) > 0.5).astype(int)
yw = make_weak_labels(y, 0.25, np.random.default_rng(6))
acc = (yw == y).mean()
assert abs(acc - 0.75) < 0.03, '准确率应≈1-flip_rate=0.75'
assert set(np.unique(yw)) <= {0, 1}
print(f'flip_rate=0.25 -> 弱标签准确率 = {acc:.3f} (≈0.75)')
print('✅ 练习 2 通过')

## ✏️ 练习 3：完整 w2s 流程 `run_w2s`

把整个流程串起来。实现 `run_w2s(Xtr, ytr, Xte, yte, flip_rate, rng)`：
用均匀噪声造弱标签 → 训强学生(弱标签) → 训天花板(真标签) → 返回 (weak_acc, student_acc, ceiling_acc, PGR)。
复用 `make_weak_labels`/`train_logreg`/`accuracy`/`compute_pgr`。

In [ ]:
def run_w2s(Xtr, ytr, Xte, yte, flip_rate, rng):
    # TODO: 1) 用 make_weak_labels 造训练集弱标签 + 测试集弱标签算 weak_acc
    #       2) train_logreg(Xtr, 弱标签) -> 学生; accuracy 得 student_acc
    #       3) train_logreg(Xtr, ytr) -> 天花板; 得 ceiling_acc
    #       4) 返回 (weak_acc, student_acc, ceiling_acc, compute_pgr(...))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
wa, sa, ca, pg = run_w2s(Xtr, ytr, Xte, yte, flip_rate=0.2, rng=np.random.default_rng(7))
print(f'弱老师={wa:.3f} 学生={sa:.3f} 天花板={ca:.3f} PGR={pg:.3f}')
assert sa > wa, 'w2s: 学生应超过老师'
assert 0.0 < pg <= 1.05, 'PGR 合理区间'
assert ca >= sa, '天花板不低于学生'
print('✅ 练习 3 通过：完整 w2s 流程跑通, 学生超过老师')

## ✏️ 练习 4：置信目标 `confidence_target`

实现置信损失的核心——混合目标。`confidence_target(weak_label, student_prob, alpha)`：
返回 `(1-alpha)*weak_label + alpha*hardened`, 其中 hardened = 1.0 若 student_prob>0.5 否则 0.0。

In [ ]:
def confidence_target(weak_label, student_prob, alpha):
    # TODO: hardened = float(student_prob>0.5); 返回混合目标
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 学生自信(prob=0.9)但弱老师标0: alpha 越大, 目标越偏向学生的1
t0 = confidence_target(0.0, 0.9, alpha=0.0)
t1 = confidence_target(0.0, 0.9, alpha=0.8)
assert t0 == 0.0, 'alpha=0 时完全听老师'
assert abs(t1 - 0.8) < 1e-9, 'alpha=0.8: 0.2*0 + 0.8*1 = 0.8(偏向学生)'
# 学生不确定(prob=0.4 -> hardened 0)且老师标0: 目标仍是0
assert confidence_target(0.0, 0.4, 0.8) == 0.0
print(f'老师标0/学生自信1: alpha=0->{t0}, alpha=0.8->{t1} (敢偏向自己)')
print('✅ 练习 4 通过：置信目标让学生在自信处偏离弱老师')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def compute_pgr(student_acc, weak_acc, ceiling_acc):
    denom = ceiling_acc - weak_acc
    if abs(denom) < 1e-12:
        return 0.0
    return (student_acc - weak_acc) / denom

In [ ]:
# 练习 2 参考答案
def make_weak_labels(y, flip_rate, rng):
    flips = rng.random(len(y)) < flip_rate
    return np.where(flips, 1 - y, y)

In [ ]:
# 练习 3 参考答案
def run_w2s(Xtr, ytr, Xte, yte, flip_rate, rng):
    wl_tr = make_weak_labels(ytr, flip_rate, rng)
    wl_te = make_weak_labels(yte, flip_rate, rng)
    weak_acc = (wl_te == yte).mean()
    ws, bs = train_logreg(Xtr, wl_tr); student_acc = accuracy(ws, bs, Xte, yte)
    wc, bc = train_logreg(Xtr, ytr);   ceiling_acc = accuracy(wc, bc, Xte, yte)
    return weak_acc, student_acc, ceiling_acc, compute_pgr(student_acc, weak_acc, ceiling_acc)

In [ ]:
# 练习 4 参考答案
def confidence_target(weak_label, student_prob, alpha):
    hardened = 1.0 if student_prob > 0.5 else 0.0
    return (1 - alpha) * weak_label + alpha * hardened

---
## 🧪 真实数据胶囊：Burns et al. 2023 的 PGR 量级

Burns et al. 在真实模型对（弱=小 GPT, 强=大 GPT）上, 跨多个 NLP 任务报告 PGR。用论文报告**量级**的代表性数字, 体会两个规律：(1) PGR 通常在 0.2–0.8; (2) 加辅助置信损失能提升 PGR。

（真实数字随任务/模型对变化, 这里用代表性量级演示趋势。）

In [ ]:
# Burns et al. 2023 趋势(量级): 不同方法的典型 PGR
RESULTS = {
    'naive (仅模仿弱标签)':       0.50,   # 朴素 w2s 已恢复约一半差距
    'with confidence loss':       0.72,   # 加置信损失显著提升
}
print(f"{'方法':>28}{'PGR':>8}")
for method, pg in RESULTS.items():
    print(f'{method:>28}{pg:>8.2f}')
# 两个规律
assert 0.2 <= RESULTS['naive (仅模仿弱标签)'] <= 0.8, 'PGR 在典型区间'
assert RESULTS['with confidence loss'] > RESULTS['naive (仅模仿弱标签)'], '置信损失提升 PGR'
improvement = RESULTS['with confidence loss'] - RESULTS['naive (仅模仿弱标签)']
print(f'\n置信损失带来的 PGR 提升 ≈ {improvement:.2f}')
print('观察: 朴素 w2s 已恢复约一半差距; 置信损失把它推得更高 —— 与本课玩具机制一致')

**🧪 胶囊练习**：实现 `pgr_to_recovered_acc(pgr, weak_acc, ceiling_acc)`：从 PGR 反推强学生的准确率。
（PGR 的逆运算：`student = weak + pgr*(ceiling-weak)`。）用它把论文的 PGR 翻译回具体准确率。

In [ ]:
def pgr_to_recovered_acc(pgr, weak_acc, ceiling_acc):
    # TODO: 返回 weak_acc + pgr*(ceiling_acc - weak_acc)
    raise NotImplementedError

In [ ]:
# 自测
# 弱老师 0.6, 天花板 0.9; PGR=0.72 对应学生准确率多少?
acc_naive = pgr_to_recovered_acc(0.50, 0.6, 0.9)
acc_conf  = pgr_to_recovered_acc(0.72, 0.6, 0.9)
assert abs(acc_naive - 0.75) < 1e-9, '0.6+0.5*0.3=0.75'
assert abs(acc_conf - 0.816) < 1e-3, '0.6+0.72*0.3=0.816'
print(f'弱老师0.6,天花板0.9: 朴素 w2s 学生≈{acc_naive:.3f}, 置信损失≈{acc_conf:.3f}')
print('✅ 胶囊练习通过：能在 PGR 与准确率之间互换')

In [ ]:
# 📖 胶囊参考答案
def pgr_to_recovered_acc(pgr, weak_acc, ceiling_acc):
    return weak_acc + pgr * (ceiling_acc - weak_acc)

### 小结
- **weak-to-strong**：用弱老师的带噪标签训强学生, 学生**超过**老师。机制 = 强先验拟合简洁真信号、容忍零散错误。
- **PGR** = (学生−老师)/(天花板−老师)：度量弱监督引出了多少强模型能力。典型 0.2–0.8。
- **辅助置信损失**：鼓励学生在自信处偏离弱老师。机制 = 更自信、更少盲从; 真实大模型上转化为 PGR 提升。
- **天花板分析**：老师越弱 PGR 越低; 师生差距过大、错误有结构、测试泄漏都会破坏 w2s。
- **意义**：对齐或许是「引出」已有能力而非「教导」新能力 —— superalignment 的乐观假设的第一个实验证据。

下一站：**模块 05 · Deliberative Alignment** —— 把监督从「事后给偏好」前移到「回答前显式推理明文规范」。